# Feature Store — Ativação de Sellers (Olist) · Produtos

Notebook de **produção** (Databricks · Spark SQL). Cada célula gera **uma feature** de forma independente, a partir dos scripts validados em `features_spark/*.sql` rodando contra `workspace.olist.*`. A **última seção** é o **tabelão consolidado** (`ativacaoOlistProdutos`): todas as features numa única tabela larga (`seller_id` × colunas), no estilo de pipeline de CTEs.

**Como usar:** importe este `.ipynb` no Databricks; rode a célula de **setup** (declara a variável de sessão `data_corte`); rode as seções. Grão de saída: **1 linha por `seller_id`** (universo = sellers com ≥1 venda antes do corte).

**Parametrização do tempo:** variável de sessão única — `DECLARE OR REPLACE VARIABLE data_corte TIMESTAMP DEFAULT TIMESTAMP'2018-07-01';` — e janelas via `datediff(data_corte, dt_venda) <= N` (D14/D28/D56/D365/Vida). Para trocar o corte, edite só o `SET VARIABLE` da célula de setup.

**Premissas-chave:** data de venda = `order_purchase_timestamp` (corte estrito `< data_corte`); **sem** filtro de `order_status`; categoria NULL → `'sem_categoria'`. Detalhes em `docs/variaveis_detalhadas.md`.

## Setup — variável de corte (`data_corte`)

Declara a **variável de sessão** `data_corte` (TIMESTAMP) e define seu valor. É o **único** ponto para trocar a data de corte: edite o `SET VARIABLE` abaixo e rode tudo. As queries leem `data_corte` diretamente e janelam por `datediff(data_corte, dt_venda) <= N`.

In [ ]:
%sql
-- Setup: variável de sessão de corte (único ponto para trocar a data).
DECLARE OR REPLACE VARIABLE data_corte TIMESTAMP DEFAULT TIMESTAMP'2018-07-01';
-- Para trocar o corte, edite a linha abaixo (ou comente-a p/ usar o DEFAULT):
SET VARIABLE data_corte = TIMESTAMP'2018-07-01';
SELECT data_corte AS corte_em_uso;

## 1. `vlCategoriasDistintas`  *(§1 — D14/D28/D56/D365/Vida)*

Quantas **categorias diferentes** o seller vendeu em cada janela. Categoria sem cadastro vira `'sem_categoria'`. Mede a **diversidade** do portfólio.

In [ ]:
%sql
WITH vendas AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        o.order_purchase_timestamp                         AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders       o ON o.order_id   = oi.order_id
    LEFT JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
)
SELECT
    seller_id,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14
                        THEN categoria END) AS vlCategoriasDistintasD14,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28
                        THEN categoria END) AS vlCategoriasDistintasD28,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56
                        THEN categoria END) AS vlCategoriasDistintasD56,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365
                        THEN categoria END) AS vlCategoriasDistintasD365,
    COUNT(DISTINCT categoria)               AS vlCategoriasDistintasVida
FROM vendas
GROUP BY seller_id;

## 2. `vlProdutosDistintos`  *(§1 — D14/D28/D56/D365/Vida)*

Quantos **`product_id` diferentes** o seller vendeu por janela. Amplitude do catálogo efetivamente vendido.

In [ ]:
%sql
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        o.order_purchase_timestamp AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < data_corte
)
SELECT
    seller_id,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14
                        THEN product_id END) AS vlProdutosDistintosD14,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28
                        THEN product_id END) AS vlProdutosDistintosD28,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56
                        THEN product_id END) AS vlProdutosDistintosD56,
    COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365
                        THEN product_id END) AS vlProdutosDistintosD365,
    COUNT(DISTINCT product_id)               AS vlProdutosDistintosVida
FROM vendas
GROUP BY seller_id;

## 3. `vlContagemCategoriaConcorrentes`  *(§2 — D14/D28/D56/D365/Vida)*

Para cada seller, quantos **outros** sellers venderam em **alguma categoria em comum**, na mesma janela. Concorrência **indireta** (substitutos). Sem concorrente na janela → 0.

In [ ]:
%sql
WITH vendas AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        o.order_purchase_timestamp                         AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders       o ON o.order_id   = oi.order_id
    LEFT JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
-- pares (seller, categoria) com a janela mais curta em que aparecem (MAX flags).
minhas_cat AS (
    SELECT seller_id, categoria,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END) AS in_d14,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END) AS in_d28,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END) AS in_d56,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END) AS in_d365
    FROM vendas
    GROUP BY seller_id, categoria
),
-- cruza A com B (mesma tabela como roster) na MESMA categoria, ambos ativos, B<>A.
concorrentes AS (
    SELECT
        a.seller_id,
        COUNT(DISTINCT CASE WHEN a.in_d14  = 1 AND b.in_d14  = 1 THEN b.seller_id END) AS q_d14,
        COUNT(DISTINCT CASE WHEN a.in_d28  = 1 AND b.in_d28  = 1 THEN b.seller_id END) AS q_d28,
        COUNT(DISTINCT CASE WHEN a.in_d56  = 1 AND b.in_d56  = 1 THEN b.seller_id END) AS q_d56,
        COUNT(DISTINCT CASE WHEN a.in_d365 = 1 AND b.in_d365 = 1 THEN b.seller_id END) AS q_d365,
        COUNT(DISTINCT b.seller_id)                                                    AS q_vida
    FROM minhas_cat a
    JOIN minhas_cat b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id
    GROUP BY a.seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT
    s.seller_id,
    COALESCE(c.q_d14,  0) AS vlContagemCategoriaConcorrentesD14,
    COALESCE(c.q_d28,  0) AS vlContagemCategoriaConcorrentesD28,
    COALESCE(c.q_d56,  0) AS vlContagemCategoriaConcorrentesD56,
    COALESCE(c.q_d365, 0) AS vlContagemCategoriaConcorrentesD365,
    COALESCE(c.q_vida, 0) AS vlContagemCategoriaConcorrentesVida
FROM spine s
LEFT JOIN concorrentes c ON c.seller_id = s.seller_id;

## 4. `vlContagemProdutosConcorrentes`  *(§2 — D14/D28/D56/D365/Vida)*

Quantos **outros** sellers venderam o **mesmo `product_id`**, por janela. Concorrência **direta** (mesmo SKU → guerra de preço). Piso de concorrência (só enxerga SKUs já unificados no catálogo).

In [ ]:
%sql
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        o.order_purchase_timestamp AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < data_corte
),
meus_produtos AS (
    SELECT seller_id, product_id,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END) AS in_d14,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END) AS in_d28,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END) AS in_d56,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END) AS in_d365
    FROM vendas
    GROUP BY seller_id, product_id
),
concorrentes AS (
    SELECT
        a.seller_id,
        COUNT(DISTINCT CASE WHEN a.in_d14  = 1 AND b.in_d14  = 1 THEN b.seller_id END) AS q_d14,
        COUNT(DISTINCT CASE WHEN a.in_d28  = 1 AND b.in_d28  = 1 THEN b.seller_id END) AS q_d28,
        COUNT(DISTINCT CASE WHEN a.in_d56  = 1 AND b.in_d56  = 1 THEN b.seller_id END) AS q_d56,
        COUNT(DISTINCT CASE WHEN a.in_d365 = 1 AND b.in_d365 = 1 THEN b.seller_id END) AS q_d365,
        COUNT(DISTINCT b.seller_id)                                                    AS q_vida
    FROM meus_produtos a
    JOIN meus_produtos b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id
    GROUP BY a.seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT
    s.seller_id,
    COALESCE(c.q_d14,  0) AS vlContagemProdutosConcorrentesD14,
    COALESCE(c.q_d28,  0) AS vlContagemProdutosConcorrentesD28,
    COALESCE(c.q_d56,  0) AS vlContagemProdutosConcorrentesD56,
    COALESCE(c.q_d365, 0) AS vlContagemProdutosConcorrentesD365,
    COALESCE(c.q_vida, 0) AS vlContagemProdutosConcorrentesVida
FROM spine s
LEFT JOIN concorrentes c ON c.seller_id = s.seller_id;

## 5. `vlCaracteresDescricao`  *(§7 — estático, Vida)*

Média, mediana, p25, p75, min e max do tamanho da descrição dos **produtos distintos** vendidos (cada SKU pesa 1). Qualidade do cadastro. **NULL → 0**: produto sem descrição conta como 0 caractere (puxa média/min).

In [ ]:
%sql
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id, COALESCE(p.product_description_lenght, 0) AS L
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
produtos AS (   -- PRODUTOS DISTINTOS (cada SKU pesa 1); TODOS entram (0 = sem descrição)
    SELECT DISTINCT seller_id, product_id, L
    FROM vendas
),
agg AS (
    SELECT
        seller_id,
        AVG(L)              AS vlMediaCaracteresDescricao,
        percentile(L, 0.50) AS vlMedianaCaracteresDescricao,
        percentile(L, 0.25) AS vl25CaracteresDescricao,
        percentile(L, 0.75) AS vl75CaracteresDescricao,
        MIN(L)              AS vlMinCaracteresDescricao,
        MAX(L)              AS vlMaxCaracteresDescricao
    FROM produtos
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT
    s.seller_id,
    agg.vlMediaCaracteresDescricao,
    agg.vlMedianaCaracteresDescricao,
    agg.vl25CaracteresDescricao,
    agg.vl75CaracteresDescricao,
    agg.vlMinCaracteresDescricao,
    agg.vlMaxCaracteresDescricao
FROM spine s
LEFT JOIN agg ON agg.seller_id = s.seller_id;

## 6. `vlMediaFotosProduto`  *(§7 — estático, Vida)*

Média de `product_photos_qty` entre os **produtos distintos** vendidos. Proxy de qualidade da vitrine. **NULL → 0**: produto sem foto conta como 0 (não há 0 'natural' na base; o mínimo real é 1).

In [ ]:
%sql
WITH vendas AS (
    SELECT oi.seller_id, oi.product_id, COALESCE(p.product_photos_qty, 0) AS fotos
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
produtos AS (   -- PRODUTOS DISTINTOS por seller (cada SKU pesa 1); TODOS entram (0 = sem foto)
    SELECT DISTINCT seller_id, product_id, fotos
    FROM vendas
),
agg AS (
    SELECT seller_id, AVG(fotos) AS vlMediaFotosProduto
    FROM produtos
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = esqueleto da entidade: 1 linha por seller (grao de saida)
SELECT s.seller_id, agg.vlMediaFotosProduto
FROM spine s
LEFT JOIN agg ON agg.seller_id = s.seller_id;

## 7. `vlPesoProduto`  *(§3 — D14/D28/D56/D365/Vida)*

Distribuição (média/mediana/p25/p75/min/max em **gramas**) do peso **por unidade vendida** + `vlTotalPesoProdutos{W}` (massa embarcada em **kg**). Ponderado por venda ("o que sai pela porta"). Peso NULL ignorado.

In [ ]:
%sql
WITH vendas AS (   -- uma linha por unidade vendida (peso pode ser NULL aqui)
    SELECT oi.seller_id,
           p.product_weight_g         AS w,
           o.order_purchase_timestamp AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
-- distribuição (por UNIDADE) + total, em cada janela. O CASE restringe a janela;
-- o peso NULL é naturalmente ignorado por AVG/percentile/MIN/MAX/SUM.
stats AS (
    SELECT seller_id,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)              AS media_d14,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END, 0.50) AS p50_d14,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END, 0.25) AS p25_d14,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END, 0.75) AS p75_d14,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)              AS mn_d14,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)              AS mx_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)/1000.0        AS tot_d14,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)              AS media_d28,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END, 0.50) AS p50_d28,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END, 0.25) AS p25_d28,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END, 0.75) AS p75_d28,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)              AS mn_d28,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)              AS mx_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)/1000.0        AS tot_d28,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)              AS media_d56,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END, 0.50) AS p50_d56,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END, 0.25) AS p25_d56,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END, 0.75) AS p75_d56,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)              AS mn_d56,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)              AS mx_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)/1000.0        AS tot_d56,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)              AS media_d365,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END, 0.50) AS p50_d365,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END, 0.25) AS p25_d365,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END, 0.75) AS p75_d365,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)              AS mn_d365,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)              AS mx_d365,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)/1000.0        AS tot_d365,
        AVG(w)              AS media_vida,
        percentile(w, 0.50) AS p50_vida,
        percentile(w, 0.25) AS p25_vida,
        percentile(w, 0.75) AS p75_vida,
        MIN(w)              AS mn_vida,
        MAX(w)              AS mx_vida,
        SUM(w)/1000.0       AS tot_vida
    FROM vendas
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = 1 linha por seller (grão de saída)
SELECT
    s.seller_id,
    st.media_d14 AS vlMediaPesoProdutoD14, st.media_d28 AS vlMediaPesoProdutoD28, st.media_d56 AS vlMediaPesoProdutoD56, st.media_d365 AS vlMediaPesoProdutoD365, st.media_vida AS vlMediaPesoProdutoVida,
    st.p50_d14 AS vlMedianaPesoProdutoD14, st.p50_d28 AS vlMedianaPesoProdutoD28, st.p50_d56 AS vlMedianaPesoProdutoD56, st.p50_d365 AS vlMedianaPesoProdutoD365, st.p50_vida AS vlMedianaPesoProdutoVida,
    st.p25_d14 AS vl25PesoProdutoD14, st.p25_d28 AS vl25PesoProdutoD28, st.p25_d56 AS vl25PesoProdutoD56, st.p25_d365 AS vl25PesoProdutoD365, st.p25_vida AS vl25PesoProdutoVida,
    st.p75_d14 AS vl75PesoProdutoD14, st.p75_d28 AS vl75PesoProdutoD28, st.p75_d56 AS vl75PesoProdutoD56, st.p75_d365 AS vl75PesoProdutoD365, st.p75_vida AS vl75PesoProdutoVida,
    st.mn_d14 AS vlMinPesoProdutoD14, st.mn_d28 AS vlMinPesoProdutoD28, st.mn_d56 AS vlMinPesoProdutoD56, st.mn_d365 AS vlMinPesoProdutoD365, st.mn_vida AS vlMinPesoProdutoVida,
    st.mx_d14 AS vlMaxPesoProdutoD14, st.mx_d28 AS vlMaxPesoProdutoD28, st.mx_d56 AS vlMaxPesoProdutoD56, st.mx_d365 AS vlMaxPesoProdutoD365, st.mx_vida AS vlMaxPesoProdutoVida,
    st.tot_d14 AS vlTotalPesoProdutosD14, st.tot_d28 AS vlTotalPesoProdutosD28, st.tot_d56 AS vlTotalPesoProdutosD56, st.tot_d365 AS vlTotalPesoProdutosD365, st.tot_vida AS vlTotalPesoProdutosVida
FROM spine s
LEFT JOIN stats st ON st.seller_id = s.seller_id;

## 8. `vlCubagemProdutos`  *(§4 — D14/D28/D56/D365/Vida)*

Média e total da cubagem (cm³ = `L×H×W`) **por unidade vendida**, janeladas (dependem de venda, crescem no tempo). Qualquer dimensão NULL → cubagem NULL (ignorada).

In [ ]:
%sql
WITH vendas AS (   -- uma linha por unidade vendida, com a cubagem do produto
    SELECT oi.seller_id,
           (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS cub,
           o.order_purchase_timestamp                                       AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
agg AS (
    SELECT seller_id,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN cub END) AS med_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN cub END) AS tot_d14,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN cub END) AS med_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN cub END) AS tot_d28,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN cub END) AS med_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN cub END) AS tot_d56,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN cub END) AS med_d365,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN cub END) AS tot_d365,
        AVG(cub) AS med_vida,
        SUM(cub) AS tot_vida
    FROM vendas
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)  -- spine = 1 linha por seller (grão de saída)
SELECT
    s.seller_id,
    a.med_d14  AS vlMediaCubagemProdutosD14,
    a.med_d28  AS vlMediaCubagemProdutosD28,
    a.med_d56  AS vlMediaCubagemProdutosD56,
    a.med_d365 AS vlMediaCubagemProdutosD365,
    a.med_vida AS vlMediaCubagemProdutosVida,
    a.tot_d14  AS vlTotalCubagemProdutosD14,
    a.tot_d28  AS vlTotalCubagemProdutosD28,
    a.tot_d56  AS vlTotalCubagemProdutosD56,
    a.tot_d365 AS vlTotalCubagemProdutosD365,
    a.tot_vida AS vlTotalCubagemProdutosVida
FROM spine s
LEFT JOIN agg a ON a.seller_id = s.seller_id;

## 9. `vlPrecoKg`  *(§5 — D14/D28/D56/D365/Vida)*

`SUM(price) / SUM(kg)` por janela (R$/kg). **Valor agregado** praticado. Mesma base (itens com peso não-nulo) no numerador e denominador; denominador 0/NULL → NULL.

In [ ]:
%sql
WITH vendas AS (   -- itens com peso não-nulo (base comum de num e den)
    SELECT oi.seller_id, oi.price AS price,
           p.product_weight_g AS w, o.order_purchase_timestamp AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
      AND p.product_weight_g IS NOT NULL
),
componentes AS (
    SELECT seller_id,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN price END)    AS receita_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)/1000.0  AS kg_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN price END)    AS receita_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)/1000.0  AS kg_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN price END)    AS receita_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)/1000.0  AS kg_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN price END)    AS receita_d365,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)/1000.0  AS kg_d365,
        SUM(price)                                                                               AS receita_vida,
        SUM(w)/1000.0                                                                            AS kg_vida
    FROM vendas
    GROUP BY seller_id
),
razoes AS (
    SELECT seller_id,
        receita_d14  / NULLIF(kg_d14,  0) AS pk_d14,
        receita_d28  / NULLIF(kg_d28,  0) AS pk_d28,
        receita_d56  / NULLIF(kg_d56,  0) AS pk_d56,
        receita_d365 / NULLIF(kg_d365, 0) AS pk_d365,
        receita_vida / NULLIF(kg_vida, 0) AS pk_vida
    FROM componentes
)
-- razão receita/kg por janela. NULL -> NULL (denominador 0/NULL via NULLIF).
SELECT
    seller_id,
    pk_d14  AS vlPrecoKgD14,
    pk_d28  AS vlPrecoKgD28,
    pk_d56  AS vlPrecoKgD56,
    pk_d365 AS vlPrecoKgD365,
    pk_vida AS vlPrecoKgVida
FROM razoes;

## 10. `vlFreteKg`  *(§5 — D14/D28/D56/D365/Vida)*

`SUM(freight_value) / SUM(kg)` por janela (R$/kg). **Custo logístico**. Idêntico ao `vlPrecoKg` trocando `price` por `freight_value`.

In [ ]:
%sql
WITH vendas AS (
    SELECT oi.seller_id, oi.freight_value AS frete,
           p.product_weight_g AS w, o.order_purchase_timestamp AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
      AND p.product_weight_g IS NOT NULL
),
componentes AS (
    SELECT seller_id,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN frete END)    AS frete_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)/1000.0  AS kg_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN frete END)    AS frete_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)/1000.0  AS kg_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN frete END)    AS frete_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)/1000.0  AS kg_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN frete END)    AS frete_d365,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)/1000.0  AS kg_d365,
        SUM(frete)                                                                               AS frete_vida,
        SUM(w)/1000.0                                                                            AS kg_vida
    FROM vendas
    GROUP BY seller_id
),
razoes AS (
    SELECT seller_id,
        frete_d14  / NULLIF(kg_d14,  0) AS fk_d14,
        frete_d28  / NULLIF(kg_d28,  0) AS fk_d28,
        frete_d56  / NULLIF(kg_d56,  0) AS fk_d56,
        frete_d365 / NULLIF(kg_d365, 0) AS fk_d365,
        frete_vida / NULLIF(kg_vida, 0) AS fk_vida
    FROM componentes
)
-- razão frete/kg por janela. NULL -> NULL (denominador 0/NULL via NULLIF).
SELECT
    seller_id,
    fk_d14  AS vlFreteKgD14,
    fk_d28  AS vlFreteKgD28,
    fk_d56  AS vlFreteKgD56,
    fk_d365 AS vlFreteKgD365,
    fk_vida AS vlFreteKgVida
FROM razoes;

## 11. `descTopCategoria{1,2,3}`  *(§6 — D14/D28/D56/D365/Vida)*

As **3 categorias mais vendidas** (por **quantidade vendida** = unidades) do seller, por janela. Desempate: unidades DESC → pedidos distintos DESC → categoria ASC. Posições inexistentes ficam `NULL`.

In [ ]:
%sql
WITH vendas AS (
    SELECT oi.seller_id,
           COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
           oi.order_id,
           o.order_purchase_timestamp                         AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders       o ON o.order_id   = oi.order_id
    LEFT JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
cat_base AS (   -- por (seller, categoria): unidades e pedidos distintos por janela
    SELECT seller_id, categoria,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END)        AS u_d14,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN order_id END) AS p_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END)        AS u_d28,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN order_id END) AS p_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END)        AS u_d56,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN order_id END) AS p_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END)        AS u_d365,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN order_id END) AS p_d365,
        COUNT(*)                                                                                       AS u_vida,
        COUNT(DISTINCT order_id)                                                                       AS p_vida
    FROM vendas
    GROUP BY seller_id, categoria
),
rk AS (
    SELECT seller_id, categoria, u_d14, u_d28, u_d56, u_d365, u_vida,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d14  DESC, p_d14  DESC, categoria ASC) AS rk_d14,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d28  DESC, p_d28  DESC, categoria ASC) AS rk_d28,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d56  DESC, p_d56  DESC, categoria ASC) AS rk_d56,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d365 DESC, p_d365 DESC, categoria ASC) AS rk_d365,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_vida DESC, p_vida DESC, categoria ASC) AS rk_vida
    FROM cat_base
)
SELECT
    seller_id,
    MAX(CASE WHEN rk_d14 =1 AND u_d14 >0 THEN categoria END) AS descTopCategoria1D14,
    MAX(CASE WHEN rk_d28 =1 AND u_d28 >0 THEN categoria END) AS descTopCategoria1D28,
    MAX(CASE WHEN rk_d56 =1 AND u_d56 >0 THEN categoria END) AS descTopCategoria1D56,
    MAX(CASE WHEN rk_d365=1 AND u_d365>0 THEN categoria END) AS descTopCategoria1D365,
    MAX(CASE WHEN rk_vida=1 AND u_vida>0 THEN categoria END) AS descTopCategoria1Vida,
    MAX(CASE WHEN rk_d14 =2 AND u_d14 >0 THEN categoria END) AS descTopCategoria2D14,
    MAX(CASE WHEN rk_d28 =2 AND u_d28 >0 THEN categoria END) AS descTopCategoria2D28,
    MAX(CASE WHEN rk_d56 =2 AND u_d56 >0 THEN categoria END) AS descTopCategoria2D56,
    MAX(CASE WHEN rk_d365=2 AND u_d365>0 THEN categoria END) AS descTopCategoria2D365,
    MAX(CASE WHEN rk_vida=2 AND u_vida>0 THEN categoria END) AS descTopCategoria2Vida,
    MAX(CASE WHEN rk_d14 =3 AND u_d14 >0 THEN categoria END) AS descTopCategoria3D14,
    MAX(CASE WHEN rk_d28 =3 AND u_d28 >0 THEN categoria END) AS descTopCategoria3D28,
    MAX(CASE WHEN rk_d56 =3 AND u_d56 >0 THEN categoria END) AS descTopCategoria3D56,
    MAX(CASE WHEN rk_d365=3 AND u_d365>0 THEN categoria END) AS descTopCategoria3D365,
    MAX(CASE WHEN rk_vida=3 AND u_vida>0 THEN categoria END) AS descTopCategoria3Vida
FROM rk
GROUP BY seller_id;

## 12. `vlShareTopCategoria{1,2,3}`  *(§6 — D14/D28/D56/D365/Vida)*

**Fração das unidades** concentrada nas 3 maiores categorias, por janela. Mede **concentração** (fragilidade). Mesmo ranking/desempate do §6; posição inexistente → `NULL` (guard `u_>0`).

In [ ]:
%sql
WITH vendas AS (
    SELECT oi.seller_id,
           COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
           oi.order_id,
           o.order_purchase_timestamp                         AS dt_venda
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders       o ON o.order_id   = oi.order_id
    LEFT JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
cat_base AS (
    SELECT seller_id, categoria,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END)        AS u_d14,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN order_id END) AS p_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END)        AS u_d28,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN order_id END) AS p_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END)        AS u_d56,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN order_id END) AS p_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END)        AS u_d365,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN order_id END) AS p_d365,
        COUNT(*)                                                                                       AS u_vida,
        COUNT(DISTINCT order_id)                                                                       AS p_vida
    FROM vendas
    GROUP BY seller_id, categoria
),
rk AS (
    SELECT seller_id, categoria, u_d14, u_d28, u_d56, u_d365, u_vida,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d14  DESC, p_d14  DESC, categoria ASC) AS rk_d14,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d28  DESC, p_d28  DESC, categoria ASC) AS rk_d28,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d56  DESC, p_d56  DESC, categoria ASC) AS rk_d56,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d365 DESC, p_d365 DESC, categoria ASC) AS rk_d365,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_vida DESC, p_vida DESC, categoria ASC) AS rk_vida
    FROM cat_base
)
SELECT
    seller_id,
    MAX(CASE WHEN rk_d14 =1 AND u_d14 >0 THEN u_d14  END) / NULLIF(SUM(u_d14),  0) AS vlShareTopCategoria1D14,
    MAX(CASE WHEN rk_d28 =1 AND u_d28 >0 THEN u_d28  END) / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria1D28,
    MAX(CASE WHEN rk_d56 =1 AND u_d56 >0 THEN u_d56  END) / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria1D56,
    MAX(CASE WHEN rk_d365=1 AND u_d365>0 THEN u_d365 END) / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria1D365,
    MAX(CASE WHEN rk_vida=1 AND u_vida>0 THEN u_vida END) / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria1Vida,
    MAX(CASE WHEN rk_d14 =2 AND u_d14 >0 THEN u_d14  END) / NULLIF(SUM(u_d14),  0) AS vlShareTopCategoria2D14,
    MAX(CASE WHEN rk_d28 =2 AND u_d28 >0 THEN u_d28  END) / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria2D28,
    MAX(CASE WHEN rk_d56 =2 AND u_d56 >0 THEN u_d56  END) / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria2D56,
    MAX(CASE WHEN rk_d365=2 AND u_d365>0 THEN u_d365 END) / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria2D365,
    MAX(CASE WHEN rk_vida=2 AND u_vida>0 THEN u_vida END) / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria2Vida,
    MAX(CASE WHEN rk_d14 =3 AND u_d14 >0 THEN u_d14  END) / NULLIF(SUM(u_d14),  0) AS vlShareTopCategoria3D14,
    MAX(CASE WHEN rk_d28 =3 AND u_d28 >0 THEN u_d28  END) / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria3D28,
    MAX(CASE WHEN rk_d56 =3 AND u_d56 >0 THEN u_d56  END) / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria3D56,
    MAX(CASE WHEN rk_d365=3 AND u_d365>0 THEN u_d365 END) / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria3D365,
    MAX(CASE WHEN rk_vida=3 AND u_vida>0 THEN u_vida END) / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria3Vida
FROM rk
GROUP BY seller_id;

## 13. `vlPesoPortfolio`  *(§8 — estático, SKU distinto)*

Perfil físico do **catálogo** do seller: distribuição (média/mediana/p25/p75/min/max em **g**) + total (**kg**) do peso por **SKU distinto** vinculado (cada `product_id` pesa 1, sem ponderar por venda, sem janela). ⚠ Contrapartida estática do §3 (peso vendido) — não confundir.

In [ ]:
%sql
WITH portfolio AS (   -- SKUs DISTINTOS vendidos pelo seller (cada product_id 1x)
    SELECT DISTINCT oi.seller_id, oi.product_id, p.product_weight_g AS w
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders   o ON o.order_id   = oi.order_id
    JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
produtos AS (   -- só SKUs com peso entram na distribuição
    SELECT seller_id, product_id, w FROM portfolio WHERE w IS NOT NULL
),
stats AS (
    SELECT seller_id,
        AVG(w)              AS media,
        percentile(w, 0.50) AS p50,
        percentile(w, 0.25) AS p25,
        percentile(w, 0.75) AS p75,
        MIN(w)              AS mn,
        MAX(w)              AS mx,
        SUM(w)/1000.0       AS total_kg
    FROM produtos
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM portfolio)  -- spine = 1 linha por seller (grão de saída)
SELECT
    s.seller_id,
    st.media    AS vlMediaPesoPortfolio,
    st.p50      AS vlMedianaPesoPortfolio,
    st.p25      AS vl25PesoPortfolio,
    st.p75      AS vl75PesoPortfolio,
    st.mn       AS vlMinPesoPortfolio,
    st.mx       AS vlMaxPesoPortfolio,
    st.total_kg AS vlTotalPesoPortfolio
FROM spine s
LEFT JOIN stats st ON st.seller_id = s.seller_id;

## ⭐ Tabelão consolidado — `ativacaoOlistProdutos`

Roda **todas as features acima numa única query** e devolve a tabela larga (`seller_id` + todas as colunas oficiais). É a montagem da feature store via `LEFT JOIN` das 13 famílias por `seller_id`, escrita como um **pipeline único de CTEs** (estilo Pierre). Validado contra os scripts individuais (0 divergências).

**Observações de montagem:** uma `base` única (1 linha por unidade vendida, com os atributos do produto) alimenta todas as CTEs; a flag `tem_produto` emula o `INNER JOIN` das famílias §3/§4/§5/§7/§8; só a §2 (concorrência) leva `COALESCE(..., 0)` para ausência, o resto fica `NULL`.

In [ ]:
%sql
WITH
-- BASE única: 1 linha por unidade vendida (order_items) antes do corte, já com
-- os atributos de produto. LEFT JOIN + `tem_produto` p/ emular o INNER das §7..§8.
base AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        oi.order_id,
        oi.price                                              AS price,
        oi.freight_value                                      AS frete,
        o.order_purchase_timestamp                            AS dt_venda,
        (p.product_id IS NOT NULL)                            AS tem_produto,
        COALESCE(p.product_category_name, 'sem_categoria')    AS categoria,     -- §1/§2/§6
        p.product_description_lenght                          AS desc_len,
        p.product_photos_qty                                  AS fotos,
        p.product_weight_g                                    AS w,
        (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS cub
    FROM workspace.olist.order_items oi
    JOIN workspace.olist.orders       o ON o.order_id   = oi.order_id
    LEFT JOIN workspace.olist.products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < data_corte
),
spine AS (SELECT DISTINCT seller_id FROM base),  -- 1 linha por seller (grão de saída)

-- =====================================================================
-- §1 — Diversidade de catálogo (01 + 02)
-- =====================================================================
f_diversidade AS (
    SELECT seller_id,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN categoria END) AS vlCategoriasDistintasD14,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN categoria END) AS vlCategoriasDistintasD28,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN categoria END) AS vlCategoriasDistintasD56,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN categoria END) AS vlCategoriasDistintasD365,
        COUNT(DISTINCT categoria)                                                                            AS vlCategoriasDistintasVida,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN product_id END) AS vlProdutosDistintosD14,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN product_id END) AS vlProdutosDistintosD28,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN product_id END) AS vlProdutosDistintosD56,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN product_id END) AS vlProdutosDistintosD365,
        COUNT(DISTINCT product_id)                                                                           AS vlProdutosDistintosVida
    FROM base
    GROUP BY seller_id
),

-- =====================================================================
-- §2 — Concorrência INDIRETA: outros sellers na MESMA CATEGORIA (03)
-- =====================================================================
cat_roster AS (
    SELECT seller_id, categoria,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END) AS in_d14,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END) AS in_d28,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END) AS in_d56,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END) AS in_d365
    FROM base GROUP BY seller_id, categoria
),
f_conc_categoria AS (
    SELECT a.seller_id,
        COUNT(DISTINCT CASE WHEN a.in_d14  = 1 AND b.in_d14  = 1 THEN b.seller_id END) AS vlContagemCategoriaConcorrentesD14,
        COUNT(DISTINCT CASE WHEN a.in_d28  = 1 AND b.in_d28  = 1 THEN b.seller_id END) AS vlContagemCategoriaConcorrentesD28,
        COUNT(DISTINCT CASE WHEN a.in_d56  = 1 AND b.in_d56  = 1 THEN b.seller_id END) AS vlContagemCategoriaConcorrentesD56,
        COUNT(DISTINCT CASE WHEN a.in_d365 = 1 AND b.in_d365 = 1 THEN b.seller_id END) AS vlContagemCategoriaConcorrentesD365,
        COUNT(DISTINCT b.seller_id)                                                    AS vlContagemCategoriaConcorrentesVida
    FROM cat_roster a
    JOIN cat_roster b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id
    GROUP BY a.seller_id
),

-- =====================================================================
-- §2 — Concorrência DIRETA: outros sellers no MESMO product_id (04)
-- =====================================================================
prod_roster AS (
    SELECT seller_id, product_id,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END) AS in_d14,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END) AS in_d28,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END) AS in_d56,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END) AS in_d365
    FROM base GROUP BY seller_id, product_id
),
f_conc_produto AS (
    SELECT a.seller_id,
        COUNT(DISTINCT CASE WHEN a.in_d14  = 1 AND b.in_d14  = 1 THEN b.seller_id END) AS vlContagemProdutosConcorrentesD14,
        COUNT(DISTINCT CASE WHEN a.in_d28  = 1 AND b.in_d28  = 1 THEN b.seller_id END) AS vlContagemProdutosConcorrentesD28,
        COUNT(DISTINCT CASE WHEN a.in_d56  = 1 AND b.in_d56  = 1 THEN b.seller_id END) AS vlContagemProdutosConcorrentesD56,
        COUNT(DISTINCT CASE WHEN a.in_d365 = 1 AND b.in_d365 = 1 THEN b.seller_id END) AS vlContagemProdutosConcorrentesD365,
        COUNT(DISTINCT b.seller_id)                                                    AS vlContagemProdutosConcorrentesVida
    FROM prod_roster a
    JOIN prod_roster b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id
    GROUP BY a.seller_id
),

-- =====================================================================
-- §7 — Caracteres da descrição (05) — Vida, PRODUTO DISTINTO, NULL -> 0
-- =====================================================================
prod_descricao AS (
    SELECT DISTINCT seller_id, product_id, COALESCE(desc_len, 0) AS L
    FROM base WHERE tem_produto
),
f_descricao AS (
    SELECT seller_id,
        AVG(L)              AS vlMediaCaracteresDescricao,
        percentile(L, 0.50) AS vlMedianaCaracteresDescricao,
        percentile(L, 0.25) AS vl25CaracteresDescricao,
        percentile(L, 0.75) AS vl75CaracteresDescricao,
        MIN(L)              AS vlMinCaracteresDescricao,
        MAX(L)              AS vlMaxCaracteresDescricao
    FROM prod_descricao GROUP BY seller_id
),

-- §7 — Média de fotos por produto (06) — Vida, PRODUTO DISTINTO, NULL -> 0
prod_fotos AS (
    SELECT DISTINCT seller_id, product_id, COALESCE(fotos, 0) AS fotos
    FROM base WHERE tem_produto
),
f_fotos AS (
    SELECT seller_id, AVG(fotos) AS vlMediaFotosProduto
    FROM prod_fotos GROUP BY seller_id
),

-- =====================================================================
-- §3 — Peso do produto VENDIDO (07): distribuição + total por UNIDADE, janelados.
-- (w já é NULL quando o produto não casou ou não tem peso -> percentile/AVG/...
--  ignoram NULL, reproduzindo o INNER JOIN + filtro de peso do script 07.)
-- =====================================================================
f_peso AS (
    SELECT seller_id,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)              AS vlMediaPesoProdutoD14,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)              AS vlMediaPesoProdutoD28,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)              AS vlMediaPesoProdutoD56,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)              AS vlMediaPesoProdutoD365,
        AVG(w)                                                                                        AS vlMediaPesoProdutoVida,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END, 0.50) AS vlMedianaPesoProdutoD14,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END, 0.50) AS vlMedianaPesoProdutoD28,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END, 0.50) AS vlMedianaPesoProdutoD56,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END, 0.50) AS vlMedianaPesoProdutoD365,
        percentile(w, 0.50)                                                                           AS vlMedianaPesoProdutoVida,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END, 0.25) AS vl25PesoProdutoD14,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END, 0.25) AS vl25PesoProdutoD28,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END, 0.25) AS vl25PesoProdutoD56,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END, 0.25) AS vl25PesoProdutoD365,
        percentile(w, 0.25)                                                                           AS vl25PesoProdutoVida,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END, 0.75) AS vl75PesoProdutoD14,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END, 0.75) AS vl75PesoProdutoD28,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END, 0.75) AS vl75PesoProdutoD56,
        percentile(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END, 0.75) AS vl75PesoProdutoD365,
        percentile(w, 0.75)                                                                           AS vl75PesoProdutoVida,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)              AS vlMinPesoProdutoD14,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)              AS vlMinPesoProdutoD28,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)              AS vlMinPesoProdutoD56,
        MIN(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)              AS vlMinPesoProdutoD365,
        MIN(w)                                                                                        AS vlMinPesoProdutoVida,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)              AS vlMaxPesoProdutoD14,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)              AS vlMaxPesoProdutoD28,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)              AS vlMaxPesoProdutoD56,
        MAX(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)              AS vlMaxPesoProdutoD365,
        MAX(w)                                                                                        AS vlMaxPesoProdutoVida,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)/1000.0        AS vlTotalPesoProdutosD14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)/1000.0        AS vlTotalPesoProdutosD28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)/1000.0        AS vlTotalPesoProdutosD56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)/1000.0        AS vlTotalPesoProdutosD365,
        SUM(w)/1000.0                                                                                  AS vlTotalPesoProdutosVida
    FROM base
    GROUP BY seller_id
),

-- =====================================================================
-- §4 — Cubagem (08): média e total por UNIDADE, janelados (cub NULL ignorado).
-- =====================================================================
f_cubagem AS (
    SELECT seller_id,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN cub END) AS vlMediaCubagemProdutosD14,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN cub END) AS vlMediaCubagemProdutosD28,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN cub END) AS vlMediaCubagemProdutosD56,
        AVG(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN cub END) AS vlMediaCubagemProdutosD365,
        AVG(cub)                                                                           AS vlMediaCubagemProdutosVida,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN cub END) AS vlTotalCubagemProdutosD14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN cub END) AS vlTotalCubagemProdutosD28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN cub END) AS vlTotalCubagemProdutosD56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN cub END) AS vlTotalCubagemProdutosD365,
        SUM(cub)                                                                           AS vlTotalCubagemProdutosVida
    FROM base
    GROUP BY seller_id
),

-- =====================================================================
-- §5 — Preço por kg (09) e Frete por kg (10): razão de totais por UNIDADE,
--      base = itens com peso não-nulo; denominador 0/NULL -> NULL.
-- =====================================================================
rs_kg_comp AS (
    SELECT seller_id,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN price END)   AS receita_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN frete END)   AS frete_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN w END)/1000.0 AS kg_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN price END)   AS receita_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN frete END)   AS frete_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN w END)/1000.0 AS kg_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN price END)   AS receita_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN frete END)   AS frete_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN w END)/1000.0 AS kg_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN price END)   AS receita_d365,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN frete END)   AS frete_d365,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN w END)/1000.0 AS kg_d365,
        SUM(price)    AS receita_vida,
        SUM(frete)    AS frete_vida,
        SUM(w)/1000.0 AS kg_vida
    FROM base
    WHERE tem_produto AND w IS NOT NULL
    GROUP BY seller_id
),
f_rs_kg AS (
    SELECT seller_id,
        receita_d14  / NULLIF(kg_d14,  0) AS vlPrecoKgD14,
        receita_d28  / NULLIF(kg_d28,  0) AS vlPrecoKgD28,
        receita_d56  / NULLIF(kg_d56,  0) AS vlPrecoKgD56,
        receita_d365 / NULLIF(kg_d365, 0) AS vlPrecoKgD365,
        receita_vida / NULLIF(kg_vida, 0) AS vlPrecoKgVida,
        frete_d14    / NULLIF(kg_d14,  0) AS vlFreteKgD14,
        frete_d28    / NULLIF(kg_d28,  0) AS vlFreteKgD28,
        frete_d56    / NULLIF(kg_d56,  0) AS vlFreteKgD56,
        frete_d365   / NULLIF(kg_d365, 0) AS vlFreteKgD365,
        frete_vida   / NULLIF(kg_vida, 0) AS vlFreteKgVida
    FROM rs_kg_comp
),

-- =====================================================================
-- §6 — Top 3 categorias (11) + Share das top 3 (12). Por unidades; desempate
--      unidades DESC -> pedidos DESC -> categoria ASC. (cat_base/rk compartilhados.)
-- =====================================================================
top_cat_base AS (
    SELECT seller_id, categoria,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN 1 ELSE 0 END)        AS u_d14,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 14  THEN order_id END) AS p_d14,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN 1 ELSE 0 END)        AS u_d28,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 28  THEN order_id END) AS p_d28,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN 1 ELSE 0 END)        AS u_d56,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 56  THEN order_id END) AS p_d56,
        SUM(CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN 1 ELSE 0 END)        AS u_d365,
        COUNT(DISTINCT CASE WHEN datediff(data_corte, dt_venda) <= 365 THEN order_id END) AS p_d365,
        COUNT(*)                                                                                       AS u_vida,
        COUNT(DISTINCT order_id)                                                                       AS p_vida
    FROM base
    GROUP BY seller_id, categoria
),
top_cat_rk AS (
    SELECT seller_id, categoria, u_d14, u_d28, u_d56, u_d365, u_vida,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d14  DESC, p_d14  DESC, categoria ASC) AS rk_d14,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d28  DESC, p_d28  DESC, categoria ASC) AS rk_d28,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d56  DESC, p_d56  DESC, categoria ASC) AS rk_d56,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_d365 DESC, p_d365 DESC, categoria ASC) AS rk_d365,
        ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY u_vida DESC, p_vida DESC, categoria ASC) AS rk_vida
    FROM top_cat_base
),
f_top_cat AS (
    SELECT seller_id,
        -- descTopCategoria{1,2,3}{W} (11)
        MAX(CASE WHEN rk_d14 =1 AND u_d14 >0 THEN categoria END) AS descTopCategoria1D14,
        MAX(CASE WHEN rk_d28 =1 AND u_d28 >0 THEN categoria END) AS descTopCategoria1D28,
        MAX(CASE WHEN rk_d56 =1 AND u_d56 >0 THEN categoria END) AS descTopCategoria1D56,
        MAX(CASE WHEN rk_d365=1 AND u_d365>0 THEN categoria END) AS descTopCategoria1D365,
        MAX(CASE WHEN rk_vida=1 AND u_vida>0 THEN categoria END) AS descTopCategoria1Vida,
        MAX(CASE WHEN rk_d14 =2 AND u_d14 >0 THEN categoria END) AS descTopCategoria2D14,
        MAX(CASE WHEN rk_d28 =2 AND u_d28 >0 THEN categoria END) AS descTopCategoria2D28,
        MAX(CASE WHEN rk_d56 =2 AND u_d56 >0 THEN categoria END) AS descTopCategoria2D56,
        MAX(CASE WHEN rk_d365=2 AND u_d365>0 THEN categoria END) AS descTopCategoria2D365,
        MAX(CASE WHEN rk_vida=2 AND u_vida>0 THEN categoria END) AS descTopCategoria2Vida,
        MAX(CASE WHEN rk_d14 =3 AND u_d14 >0 THEN categoria END) AS descTopCategoria3D14,
        MAX(CASE WHEN rk_d28 =3 AND u_d28 >0 THEN categoria END) AS descTopCategoria3D28,
        MAX(CASE WHEN rk_d56 =3 AND u_d56 >0 THEN categoria END) AS descTopCategoria3D56,
        MAX(CASE WHEN rk_d365=3 AND u_d365>0 THEN categoria END) AS descTopCategoria3D365,
        MAX(CASE WHEN rk_vida=3 AND u_vida>0 THEN categoria END) AS descTopCategoria3Vida,
        -- vlShareTopCategoria{1,2,3}{W} (12) — / em Spark já promove INT->DOUBLE
        MAX(CASE WHEN rk_d14 =1 AND u_d14 >0 THEN u_d14  END) / NULLIF(SUM(u_d14),  0) AS vlShareTopCategoria1D14,
        MAX(CASE WHEN rk_d28 =1 AND u_d28 >0 THEN u_d28  END) / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria1D28,
        MAX(CASE WHEN rk_d56 =1 AND u_d56 >0 THEN u_d56  END) / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria1D56,
        MAX(CASE WHEN rk_d365=1 AND u_d365>0 THEN u_d365 END) / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria1D365,
        MAX(CASE WHEN rk_vida=1 AND u_vida>0 THEN u_vida END) / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria1Vida,
        MAX(CASE WHEN rk_d14 =2 AND u_d14 >0 THEN u_d14  END) / NULLIF(SUM(u_d14),  0) AS vlShareTopCategoria2D14,
        MAX(CASE WHEN rk_d28 =2 AND u_d28 >0 THEN u_d28  END) / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria2D28,
        MAX(CASE WHEN rk_d56 =2 AND u_d56 >0 THEN u_d56  END) / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria2D56,
        MAX(CASE WHEN rk_d365=2 AND u_d365>0 THEN u_d365 END) / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria2D365,
        MAX(CASE WHEN rk_vida=2 AND u_vida>0 THEN u_vida END) / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria2Vida,
        MAX(CASE WHEN rk_d14 =3 AND u_d14 >0 THEN u_d14  END) / NULLIF(SUM(u_d14),  0) AS vlShareTopCategoria3D14,
        MAX(CASE WHEN rk_d28 =3 AND u_d28 >0 THEN u_d28  END) / NULLIF(SUM(u_d28),  0) AS vlShareTopCategoria3D28,
        MAX(CASE WHEN rk_d56 =3 AND u_d56 >0 THEN u_d56  END) / NULLIF(SUM(u_d56),  0) AS vlShareTopCategoria3D56,
        MAX(CASE WHEN rk_d365=3 AND u_d365>0 THEN u_d365 END) / NULLIF(SUM(u_d365), 0) AS vlShareTopCategoria3D365,
        MAX(CASE WHEN rk_vida=3 AND u_vida>0 THEN u_vida END) / NULLIF(SUM(u_vida), 0) AS vlShareTopCategoria3Vida
    FROM top_cat_rk
    GROUP BY seller_id
),

-- =====================================================================
-- §8 — Peso do PORTFÓLIO (13): SKU DISTINTO vinculado, estático (sem janela).
-- =====================================================================
portfolio AS (
    SELECT DISTINCT seller_id, product_id, w
    FROM base WHERE tem_produto AND w IS NOT NULL
),
f_portfolio AS (
    SELECT seller_id,
        AVG(w)              AS vlMediaPesoPortfolio,
        percentile(w, 0.50) AS vlMedianaPesoPortfolio,
        percentile(w, 0.25) AS vl25PesoPortfolio,
        percentile(w, 0.75) AS vl75PesoPortfolio,
        MIN(w)              AS vlMinPesoPortfolio,
        MAX(w)              AS vlMaxPesoPortfolio,
        SUM(w)/1000.0       AS vlTotalPesoPortfolio
    FROM portfolio GROUP BY seller_id
)

-- =====================================================================
-- TABELÃO FINAL — spine LEFT JOIN de todas as famílias por seller_id.
-- COALESCE só onde o script original devolve 0 p/ ausência (§2 concorrência);
-- demais ausências permanecem NULL (sem produto/peso/categoria na janela).
-- =====================================================================
SELECT
    s.seller_id,

    -- §1 Diversidade de catálogo
    d.vlCategoriasDistintasD14, d.vlCategoriasDistintasD28, d.vlCategoriasDistintasD56, d.vlCategoriasDistintasD365, d.vlCategoriasDistintasVida,
    d.vlProdutosDistintosD14,   d.vlProdutosDistintosD28,   d.vlProdutosDistintosD56,   d.vlProdutosDistintosD365,   d.vlProdutosDistintosVida,

    -- §2 Concorrência indireta (mesma categoria) — ausência -> 0
    COALESCE(cc.vlContagemCategoriaConcorrentesD14,  0) AS vlContagemCategoriaConcorrentesD14,
    COALESCE(cc.vlContagemCategoriaConcorrentesD28,  0) AS vlContagemCategoriaConcorrentesD28,
    COALESCE(cc.vlContagemCategoriaConcorrentesD56,  0) AS vlContagemCategoriaConcorrentesD56,
    COALESCE(cc.vlContagemCategoriaConcorrentesD365, 0) AS vlContagemCategoriaConcorrentesD365,
    COALESCE(cc.vlContagemCategoriaConcorrentesVida, 0) AS vlContagemCategoriaConcorrentesVida,

    -- §2 Concorrência direta (mesmo product_id) — ausência -> 0
    COALESCE(cp.vlContagemProdutosConcorrentesD14,  0) AS vlContagemProdutosConcorrentesD14,
    COALESCE(cp.vlContagemProdutosConcorrentesD28,  0) AS vlContagemProdutosConcorrentesD28,
    COALESCE(cp.vlContagemProdutosConcorrentesD56,  0) AS vlContagemProdutosConcorrentesD56,
    COALESCE(cp.vlContagemProdutosConcorrentesD365, 0) AS vlContagemProdutosConcorrentesD365,
    COALESCE(cp.vlContagemProdutosConcorrentesVida, 0) AS vlContagemProdutosConcorrentesVida,

    -- §7 Caracteres da descrição (Vida) + média de fotos (Vida)
    dc.vlMediaCaracteresDescricao, dc.vlMedianaCaracteresDescricao, dc.vl25CaracteresDescricao,
    dc.vl75CaracteresDescricao, dc.vlMinCaracteresDescricao, dc.vlMaxCaracteresDescricao,
    ft.vlMediaFotosProduto,

    -- §3 Peso do produto vendido (distribuição + total por UNIDADE, 5 janelas)
    pe.vlMediaPesoProdutoD14, pe.vlMediaPesoProdutoD28, pe.vlMediaPesoProdutoD56, pe.vlMediaPesoProdutoD365, pe.vlMediaPesoProdutoVida,
    pe.vlMedianaPesoProdutoD14, pe.vlMedianaPesoProdutoD28, pe.vlMedianaPesoProdutoD56, pe.vlMedianaPesoProdutoD365, pe.vlMedianaPesoProdutoVida,
    pe.vl25PesoProdutoD14, pe.vl25PesoProdutoD28, pe.vl25PesoProdutoD56, pe.vl25PesoProdutoD365, pe.vl25PesoProdutoVida,
    pe.vl75PesoProdutoD14, pe.vl75PesoProdutoD28, pe.vl75PesoProdutoD56, pe.vl75PesoProdutoD365, pe.vl75PesoProdutoVida,
    pe.vlMinPesoProdutoD14, pe.vlMinPesoProdutoD28, pe.vlMinPesoProdutoD56, pe.vlMinPesoProdutoD365, pe.vlMinPesoProdutoVida,
    pe.vlMaxPesoProdutoD14, pe.vlMaxPesoProdutoD28, pe.vlMaxPesoProdutoD56, pe.vlMaxPesoProdutoD365, pe.vlMaxPesoProdutoVida,
    pe.vlTotalPesoProdutosD14, pe.vlTotalPesoProdutosD28, pe.vlTotalPesoProdutosD56, pe.vlTotalPesoProdutosD365, pe.vlTotalPesoProdutosVida,

    -- §4 Cubagem (média + total por UNIDADE, 5 janelas)
    cb.vlMediaCubagemProdutosD14, cb.vlMediaCubagemProdutosD28, cb.vlMediaCubagemProdutosD56, cb.vlMediaCubagemProdutosD365, cb.vlMediaCubagemProdutosVida,
    cb.vlTotalCubagemProdutosD14, cb.vlTotalCubagemProdutosD28, cb.vlTotalCubagemProdutosD56, cb.vlTotalCubagemProdutosD365, cb.vlTotalCubagemProdutosVida,

    -- §5 Preço/kg e Frete/kg
    rk.vlPrecoKgD14, rk.vlPrecoKgD28, rk.vlPrecoKgD56, rk.vlPrecoKgD365, rk.vlPrecoKgVida,
    rk.vlFreteKgD14, rk.vlFreteKgD28, rk.vlFreteKgD56, rk.vlFreteKgD365, rk.vlFreteKgVida,

    -- §6 Top 3 categorias (nome)
    tc.descTopCategoria1D14, tc.descTopCategoria1D28, tc.descTopCategoria1D56, tc.descTopCategoria1D365, tc.descTopCategoria1Vida,
    tc.descTopCategoria2D14, tc.descTopCategoria2D28, tc.descTopCategoria2D56, tc.descTopCategoria2D365, tc.descTopCategoria2Vida,
    tc.descTopCategoria3D14, tc.descTopCategoria3D28, tc.descTopCategoria3D56, tc.descTopCategoria3D365, tc.descTopCategoria3Vida,

    -- §6 Share das top 3 categorias
    tc.vlShareTopCategoria1D14, tc.vlShareTopCategoria1D28, tc.vlShareTopCategoria1D56, tc.vlShareTopCategoria1D365, tc.vlShareTopCategoria1Vida,
    tc.vlShareTopCategoria2D14, tc.vlShareTopCategoria2D28, tc.vlShareTopCategoria2D56, tc.vlShareTopCategoria2D365, tc.vlShareTopCategoria2Vida,
    tc.vlShareTopCategoria3D14, tc.vlShareTopCategoria3D28, tc.vlShareTopCategoria3D56, tc.vlShareTopCategoria3D365, tc.vlShareTopCategoria3Vida,

    -- §8 Peso do portfólio (estático, SKU distinto)
    pf.vlMediaPesoPortfolio, pf.vlMedianaPesoPortfolio, pf.vl25PesoPortfolio, pf.vl75PesoPortfolio,
    pf.vlMinPesoPortfolio, pf.vlMaxPesoPortfolio, pf.vlTotalPesoPortfolio

FROM spine s
LEFT JOIN f_diversidade    d  ON d.seller_id  = s.seller_id
LEFT JOIN f_conc_categoria cc ON cc.seller_id = s.seller_id
LEFT JOIN f_conc_produto   cp ON cp.seller_id = s.seller_id
LEFT JOIN f_descricao      dc ON dc.seller_id = s.seller_id
LEFT JOIN f_fotos          ft ON ft.seller_id = s.seller_id
LEFT JOIN f_peso           pe ON pe.seller_id = s.seller_id
LEFT JOIN f_cubagem        cb ON cb.seller_id = s.seller_id
LEFT JOIN f_rs_kg          rk ON rk.seller_id = s.seller_id
LEFT JOIN f_top_cat        tc ON tc.seller_id = s.seller_id
LEFT JOIN f_portfolio      pf ON pf.seller_id = s.seller_id;